# Session 2 — Data Cleaning
### FAA National Wildlife Strike Database (`Public.csv`)

Session 1 profiled the raw extract without changing anything. This notebook applies the specific fixes
that profiling pass actually built evidence for, each with its own reasoning documented before the code,
and writes the result to a new file. **`data/procured/Public.csv` is never modified;** this notebook only
reads it and writes a separate cleaned copy.

*Revised in session 3:* the session 2 review found that `01_data_profiling.ipynb`'s `TIME` fill-rate
measurement counted whitespace-only strings as present, which inflated its apparent completeness in
2024–2026 by up to 45 points. Step 0 below (added this session, applied before everything else) fixes the
underlying data issue that measurement error was reacting to. The review also caught a false claim in
Step 6's reasoning and a tautological validation check in the same step — both corrected in place, not
just noted.

What gets fixed this session, and why each one is evidenced rather than assumed:
0. Whitespace-only strings, across every string column — converted to real nulls (added session 3; see
   below for why this has to run first).
1. `NR_INJURIES` / `NR_FATALITIES` — blank filled with 0 (confirmed: every non-null value in both fields
   is >=1, across all 354,144 rows).
2. `TRANSFER` — dropped (confirmed constant; FAA's own field dictionary calls it unused).
3. `DAMAGE_LEVEL` — blank recoded to `"Unknown"` (FAA's dictionary defines blank as a real sixth level
   here, not missing data).
4. `NUM_SEEN` / `NUM_STRUCK` — cast to an explicit ordered categorical (currently plain strings that sort
   wrong).
5. `SPEED`, `INDEX_NR 1867606` — corrected from `2700` to `280`, the one row with direct first-person
   evidence of what the value should have been.
6. `LATITUDE` (and, incidentally, `LONGITUDE`) — the ~15 corrupted values flagged in the original
   diagnosis but not yet investigated, resolved here.

Three things flagged in session 1 are **deliberately not touched this session** — covered in their own
section below, with the reasoning for leaving each alone stated as explicitly as the reasoning for fixing
the steps above.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 140)
pd.set_option('display.max_colwidth', 80)

df = pd.read_csv('../data/procured/Public.csv', low_memory=False)
n_rows_before = len(df)
n_cols_before = df.shape[1]
print('loaded:', df.shape)


loaded: (354144, 102)


## Step 0 — Whitespace-only strings (added session 3)

**Issue:** session 3 found that `01_data_profiling.ipynb` measured `TIME`'s fill rate with `.notna()`,
which counts a whitespace-only string (`'     '`) as present. `TIME` turns out to have 31,298 such rows —
no real information, counted as data anyway. That's a measurement error, not a `TIME`-specific one in
principle, so before deciding how big a problem it is, it needs a sweep across every string column rather
than a fix aimed only at the column where it was first noticed.

In [2]:
str_cols = df.select_dtypes(include=['str']).columns
print(f'string-typed columns to check: {len(str_cols)}')
print()

ws_rows = []
for c in str_cols:
    s = df[c]
    n_notna = s.notna().sum()
    if n_notna == 0:
        continue
    n_ws = (s.notna() & (s.str.strip() == '')).sum()
    if n_ws > 0:
        ws_rows.append({'column': c, 'n_whitespace_only': n_ws,
                         'pct_of_non_null': round(100 * n_ws / n_notna, 2)})

ws_table = pd.DataFrame(ws_rows).sort_values('n_whitespace_only', ascending=False)
print(ws_table.to_string(index=False))


string-typed columns to check: 42

       column  n_whitespace_only  pct_of_non_null
         TIME              31298            13.44
     LOCATION                  4             0.01
          FLT                  3             0.00
      REMARKS                  2             0.00
OTHER_SPECIFY                  1             0.00


**Decision:** convert whitespace-only strings to proper `NaN` across every string-typed column,
applied first — before any of the six steps below — so everything downstream sees genuine nulls rather
than content-free strings that test as "present."

**Why:** the sweep confirms the problem is overwhelmingly `TIME` (31,298 of 31,308 total occurrences,
99.97%) but not exclusively — `LOCATION` (4), `FLT` (3), `REMARKS` (2), and `OTHER_SPECIFY` (1) each have
a handful too. Fixing only `TIME` would leave the same latent bug sitting in four other columns, waiting
to distort whatever notebook touches them next. This doesn't change what's *true* about the data — a
whitespace-only entry was never real information — it just makes that already-true fact visible to code
that tests for missingness. That's different from imputation: nothing here guesses at what a blank `TIME`
value should have been, it just stops miscounting it as known.

In [3]:
before_notna = {c: df[c].notna().sum() for c in str_cols}

df[str_cols] = df[str_cols].replace(r'^\s*$', None, regex=True)

print('rows changed per column:')
total_changed = 0
for c in str_cols:
    diff = before_notna[c] - df[c].notna().sum()
    if diff > 0:
        print(f'  {c}: {diff}')
        total_changed += diff
print('total:', total_changed)
print()
print('TIME notna before -> after:', before_notna['TIME'], '->', df['TIME'].notna().sum())
print('row count unchanged:', len(df) == n_rows_before)


rows changed per column:
  TIME: 31298
  LOCATION: 4
  FLT: 3
  OTHER_SPECIFY: 1
  REMARKS: 2
total: 31308

TIME notna before -> after: 232822 -> 201524
row count unchanged: True


## Step 1 — `NR_INJURIES` / `NR_FATALITIES`

**Issue:** both fields are ~99.9% blank. Read naively, that looks like a data-quality gap.

**Decision:** `fillna(0)`, then cast to integer.

**Why:** session 1 checked every non-null value in both fields across all 354,144 rows — the minimum
value found was 1, with no zeroes anywhere. That's only possible if the form/database only populates
these fields when there *was* an injury or fatality to report. Blank reliably means "none happened," not
"nobody answered" — so filling with 0 encodes what the data already means, it doesn't invent a value.

In [4]:
df['NR_INJURIES'] = df['NR_INJURIES'].fillna(0).astype(int)
df['NR_FATALITIES'] = df['NR_FATALITIES'].fillna(0).astype(int)

print('NR_INJURIES:  dtype', df['NR_INJURIES'].dtype, '| nulls', df['NR_INJURIES'].isna().sum(),
      '| value counts:')
print(df['NR_INJURIES'].value_counts().sort_index())
print()
print('NR_FATALITIES: dtype', df['NR_FATALITIES'].dtype, '| nulls', df['NR_FATALITIES'].isna().sum())


NR_INJURIES:  dtype int64 | nulls 0 | value counts:
NR_INJURIES
0    353840
1       237
2        59
3         3
4         2
5         2
7         1
Name: count, dtype: int64

NR_FATALITIES: dtype int64 | nulls 0


## Step 2 — `TRANSFER`

**Issue:** constant across all 354,144 rows.

**Decision:** drop the column entirely.

**Why:** session 1 confirmed it's always `0` with no exceptions, and the FAA's own field dictionary
describes it as "unused at this time." A column with one value in every row carries no information for
any future analysis — keeping it around is pure clutter, not caution.

In [5]:
assert df['TRANSFER'].nunique(dropna=False) == 1
df = df.drop(columns=['TRANSFER'])
print('columns:', n_cols_before, '->', df.shape[1])


columns: 102 -> 101


## Step 3 — `DAMAGE_LEVEL`

**Issue:** 35.27% blank (`NaN`).

**Decision:** recode blank to the explicit string `"Unknown"`.

**Why:** the FAA field dictionary defines blank as meaning `"Unknown"` here — a real sixth category,
semantically parallel to `N`/`M`/`M?`/`S`/`D`, not a gap in reporting the way `HEIGHT` or `SPEED`
missingness is. Leaving it `NaN` misrepresents a defined category as absent data.

**Caution for later:** once this runs, `.isna()` on `DAMAGE_LEVEL` will behave differently than it did in
session 1 — there's nothing left for it to find. Any code that used `DAMAGE_LEVEL.isna()` as a stand-in
for "damage unknown" needs to check `== "Unknown"` against the cleaned data instead.

In [6]:
df['DAMAGE_LEVEL'] = df['DAMAGE_LEVEL'].fillna('Unknown')
print(df['DAMAGE_LEVEL'].value_counts(dropna=False))


DAMAGE_LEVEL
N          207045
Unknown    124897
M?           8903
M            8795
S            4413
D              91
Name: count, dtype: int64


## Step 4 — `NUM_SEEN` / `NUM_STRUCK`

**Issue:** both fields hold the four ordinal bucket strings (`"1"`, `"2-10"`, `"11-100"`,
`"More than 100"`), stored as plain text.

**Decision:** cast both to `pd.CategoricalDtype(ordered=True)` with the buckets in their real order.

**Why:** plain strings compare and sort lexicographically, not numerically — `"11-100"` sorts before
`"2-10"` because `"1"` < `"2"` as characters, which is backwards from what the buckets mean. An explicit
ordered categorical fixes every future sort, `groupby`, or comparison on this column at once, instead of
leaving it as a trap for whoever uses it next.

In [7]:
print('plain string sort of the four bucket labels (wrong order):')
print(' ', sorted(['1', '2-10', '11-100', 'More than 100']))

ordinal_dtype = pd.CategoricalDtype(categories=['1', '2-10', '11-100', 'More than 100'], ordered=True)
df['NUM_SEEN'] = df['NUM_SEEN'].astype(ordinal_dtype)
df['NUM_STRUCK'] = df['NUM_STRUCK'].astype(ordinal_dtype)

print()
print('as an ordered categorical, the real order is preserved:', list(df['NUM_SEEN'].cat.categories))
print('and comparisons now respect it (vectorized, column-level):')
print('  NUM_STRUCK < "11-100" :', (df['NUM_STRUCK'] < '11-100').sum(), 'rows correctly below that bucket')
print('  NUM_STRUCK < "2-10"   :', (df['NUM_STRUCK'] < '2-10').sum(), 'rows correctly below that bucket')


plain string sort of the four bucket labels (wrong order):
  ['1', '11-100', '2-10', 'More than 100']

as an ordered categorical, the real order is preserved: ['1', '2-10', '11-100', 'More than 100']
and comparisons now respect it (vectorized, column-level):
  NUM_STRUCK < "11-100" : 351664 rows correctly below that bucket
  NUM_STRUCK < "2-10"   : 315649 rows correctly below that bucket


One thing worth knowing if you build on this later: the fix holds for *column-level* comparisons
like the ones above (and for `.sort_values()`, `groupby`, etc.) — that's how this field will actually get
used. It does **not** hold if you pull a single value out with something like `.iloc[0]` and compare two
bare scalars with `<` — pandas hands back a plain Python `str` at that point, dropping the category
ordering, and you're back to lexicographic comparison. Checked directly rather than assumed: worth
knowing before it produces a confusing bug three notebooks from now.

## Step 5 — `SPEED`, `INDEX_NR 1867606`

**Issue:** recorded as `2700` knots — roughly 4x the speed of sound, impossible for any aircraft in this
dataset.

**Decision:** correct to `280`.

**Why:** this is the one row from session 1's outlier pass with direct first-person evidence of the
correct value — the pilot's own remarks state *"we elected to cruise at FL280 and 280 knots."* This isn't
an inferred correction; it's transcribing what the source material already says. (`SPEED = 541`,
`INDEX_NR 1476144`, is a different row with a similarly implausible value but no equivalent direct
evidence — left alone, see below.)

In [8]:
mask = df['INDEX_NR'] == 1867606
assert mask.sum() == 1
before_val = df.loc[mask, 'SPEED'].iloc[0]
df.loc[mask, 'SPEED'] = 280
print(f'INDEX_NR 1867606: SPEED {before_val} -> {df.loc[mask, "SPEED"].iloc[0]}')


INDEX_NR 1867606: SPEED 2700.0 -> 280.0


## Step 6 — `LATITUDE` (and `LONGITUDE`)

**Issue:** flagged in the original project diagnosis but never actually investigated until now — "15
corrupted values out of ~306k non-null (a few in degree-minute-second format, a few that look like a full
'lat,long' pair jammed into the latitude field alone)." Time to find out exactly what that means row by
row rather than take the summary on faith.

In [9]:
def parses_as_float(x):
    try:
        float(x)
        return True
    except (ValueError, TypeError):
        return False

lat_raw = df['LATITUDE']
lon_raw = df['LONGITUDE']
bad_mask = lat_raw.notna() & ~lat_raw.apply(parses_as_float)
print('corrupted LATITUDE values found:', bad_mask.sum())
print()
print(df.loc[bad_mask, ['INDEX_NR', 'AIRPORT_ID', 'LATITUDE', 'LONGITUDE']].to_string(index=False))


corrupted LATITUDE values found: 15

 INDEX_NR AIRPORT_ID       LATITUDE      LONGITUDE
  1408343       LKTB    49°09'05?N      16°41'40?E
  1410435       LTFM 41°15'44?N 28°            NaN
  1427099       PALP 70.3442778,-15     -150.94475
  1550527       PAVE 67.0086750,-14   -146.3663583
  1571936       KBUU 42.6907171,-88 42.6907171,-88
  1646857       LTFM 41°15'44?N 28°            NaN
  1648448       LTFM 41°15'44?N 28°            NaN
  1691454       KBKT 37.0747673,-77    -77.9565984
  1709852       EPLB 51°14'25.00?N  022°42'49.00?E
  1787810       PALP 70.3442778,-15     -150.94475
  1800969       KBKT 37.0747673,-77    -77.9565984
  1801711       5PA5 40.1821881,-79 40.1821881,-79
  1858956       KBKT 37.0747673,-77    -77.9565984
  1863624       LTFM 41°15'44?N 28°            NaN
  1888287       LTFM 41°15'44?N 28°            NaN


Reading all 15 individually (rather than pattern-matching blindly, per the session brief) points to
one root cause with two different surface appearances, not two independent phenomena:

- **A hard ~14-character limit on the field, truncating anything longer regardless of what was typed.**
  14 of the 15 corrupted values are *exactly* 14 characters — including the five `LTFM` DMS rows
  (`41°15'44?N 28°`) and `EPLB` (`51°14'25.00?N `, trailing space included), not just the decimal ones.
  The one exception, `LKTB` at 11 characters, is short enough to have survived intact — which is itself
  evidence for the length-limit theory rather than against it: the one value under the apparent ceiling is
  also the one value that isn't cut off.
- **What got typed determines what's recoverable, not the truncation itself.** Two rows (`LKTB`, `EPLB`)
  were writing degree-minute-second coordinates (`49°09'05?N` — the `?` stands in for a mangled `'`/`"`
  mark, almost certainly a character-encoding slip) and fit inside 14 characters in both `LATITUDE` and
  `LONGITUDE`, so both recover fully. Five rows (`LTFM`, all Istanbul Airport) were also writing DMS but
  ran long enough that only `LATITUDE` survives complete, with a truncated, incomplete longitude fragment
  (`28°`, no minutes/seconds/direction) crammed into the same field — `LATITUDE` recovers, `LONGITUDE`
  doesn't (and was already `NaN` in the real `LONGITUDE` column). Eight rows were writing decimal
  `lat,long` pairs instead (`70.3442778,-150.94475`, truncated to `70.3442778,-15`): for 6 of them (`PALP`
  x2, `PAVE`, `KBKT` x3) the real `LONGITUDE` column independently holds the complete, correctly-signed
  value, so only `LATITUDE` needs recovering; for 2 (`KBUU`, `5PA5`), `LONGITUDE` holds the *identical*
  truncated fragment as `LATITUDE` rather than a distinct value — the real longitude was overwritten, not
  just truncated to something usable, so it isn't recoverable from this row.

**Decision:** parse the DMS and comma-jammed values back to decimal degrees where the row's own data makes
that unambiguous; null out the two `LONGITUDE` values that were overwritten with a duplicate fragment
rather than truncated to something usable.

**Considered and rejected — and the rejection is stronger than "out of scope."** The instinct is to
cross-reference other rows for the same `AIRPORT_ID` to fill the unrecoverable `LTFM` longitudes. That
turns out not to be merely out of scope, it's impossible from inside this dataset: `LTFM` appears exactly
5 times total, and all 5 are the corrupted rows being fixed right now — there is no uncorrupted `LTFM` row
to borrow from. The same holds for every other airport in this set: `LKTB` 1 row, `EPLB` 1, `PALP` 2,
`PAVE` 1, `KBKT` 3, `KBUU` 1, `5PA5` 1 — 15 rows, 15 corrupted, zero uncorrupted siblings anywhere. (Even
setting that aside, cross-referencing would still be imputation from other data rather than parsing this
row's own entry — a materially different kind of decision. But the dataset doesn't offer the choice
either way.) Worth a note for later — an external gazetteer could still resolve these — not a fix from
inside this dataset.

In [10]:
def parse_dms(s):
    """Parse a degree-minute-second string like 49°09'05?N into signed decimal
    degrees. The '?' stands in for a corrupted prime/double-prime mark; seconds are
    optional (EPLB's values have them, LKTB's/LTFM's don't)."""
    m = re.match(r"(\d+)\xb0(\d+)'([\d.]+)?.?([NSEW])", str(s).strip())
    if not m:
        return None
    deg, minute, sec, direction = m.groups()
    val = float(deg) + float(minute) / 60 + (float(sec) if sec else 0.0) / 3600
    return round(-val, 6) if direction in ('S', 'W') else round(val, 6)

# pandas 3's dedicated string dtype refuses to accept a raw float assigned into a
# still-'str'-typed Series (unlike legacy 'object' dtype, which allowed it silently) --
# this is the concrete case the aside in the session 1 notebook was flagging. Casting
# to object first sidesteps it; both columns go to float64 once every value is numeric.
new_lat = lat_raw.astype(object).copy()
new_lon = lon_raw.astype(object).copy()

for idx in df.index[bad_mask]:
    raw_lat, raw_lon = lat_raw.at[idx], lon_raw.at[idx]
    if '\xb0' in raw_lat:
        new_lat.at[idx] = parse_dms(raw_lat)
        if isinstance(raw_lon, str) and '\xb0' in raw_lon:
            new_lon.at[idx] = parse_dms(raw_lon)
        # else: longitude fragment unrecoverable -- already NaN in every such case, left as-is
    elif ',' in raw_lat:
        new_lat.at[idx] = float(raw_lat.split(',')[0])
        if raw_lon == raw_lat:
            new_lon.at[idx] = np.nan  # duplicated truncated fragment, not a real value
        # else: longitude already holds a distinct, complete, correctly-signed value -- leave it

n_lat_null_before, n_lon_null_before = lat_raw.isna().sum(), lon_raw.isna().sum()

df['LATITUDE'] = new_lat.astype(float)
df['LONGITUDE'] = new_lon.astype(float)

# Real checks, not "is the column float64" (true by construction the moment .astype(float)
# succeeds, so it can never fail regardless of whether the parse was correct -- caught in
# session 3's review). A silent parse failure would show up as either a new null that
# shouldn't be there, or a value outside a physically valid coordinate range.
n_lat_null_after, n_lon_null_after = df['LATITUDE'].isna().sum(), df['LONGITUDE'].isna().sum()
print('LATITUDE nulls: before', n_lat_null_before, '-> after', n_lat_null_after,
      f'(delta {n_lat_null_after - n_lat_null_before}, expected 0)')
print('LONGITUDE nulls: before', n_lon_null_before, '-> after', n_lon_null_after,
      f'(delta {n_lon_null_after - n_lon_null_before}, expected +2, for KBUU/5PA5)')
assert n_lat_null_after - n_lat_null_before == 0
assert n_lon_null_after - n_lon_null_before == 2

lat_in_range = df['LATITUDE'].dropna().between(-90, 90).all()
lon_in_range = df['LONGITUDE'].dropna().between(-180, 180).all()
print(f"LATITUDE range: [{df['LATITUDE'].min():.4f}, {df['LATITUDE'].max():.4f}]  -- within [-90,90]: {lat_in_range}")
print(f"LONGITUDE range: [{df['LONGITUDE'].min():.4f}, {df['LONGITUDE'].max():.4f}]  -- within [-180,180]: {lon_in_range}")
assert lat_in_range and lon_in_range
print()
print(df.loc[bad_mask, ['INDEX_NR', 'AIRPORT_ID', 'LATITUDE', 'LONGITUDE']].to_string(index=False))


LATITUDE nulls: before 48037 -> after 48037 (delta 0, expected 0)
LONGITUDE nulls: before 48042 -> after 48044 (delta 2, expected +2, for KBUU/5PA5)
LATITUDE range: [-38.0406, 71.2854]  -- within [-90,90]: True
LONGITUDE range: [-177.3810, 178.5592]  -- within [-180,180]: True

 INDEX_NR AIRPORT_ID  LATITUDE   LONGITUDE
  1408343       LKTB 49.151389   16.694444
  1410435       LTFM 41.262222         NaN
  1427099       PALP 70.344278 -150.944750
  1550527       PAVE 67.008675 -146.366358
  1571936       KBUU 42.690717         NaN
  1646857       LTFM 41.262222         NaN
  1648448       LTFM 41.262222         NaN
  1691454       KBKT 37.074767  -77.956598
  1709852       EPLB 51.240278   22.713611
  1787810       PALP 70.344278 -150.944750
  1800969       KBKT 37.074767  -77.956598
  1801711       5PA5 40.182188         NaN
  1858956       KBKT 37.074767  -77.956598
  1863624       LTFM 41.262222         NaN
  1888287       LTFM 41.262222         NaN


All 15 resolve cleanly. As a sanity check beyond "it parsed without crashing": the recovered
coordinates land where the airport names say they should — `LKTB` (Brno-Tuřany, Czech Republic) at
~49.15°N 16.69°E, `EPLB` (Lublin, Poland) at ~51.24°N 22.71°E, the two Alaska airstrips
(`PALP`, `PAVE`) both north of 67°N, `KBKT` (Blackstone, Virginia) at ~37.07°N. Nothing here
required outside data to check — internal geographic plausibility was enough.

## Explicitly left alone this session

A documented decision not to act is still a decision — recorded here with the same one-line-reason
standard as the six fixes above, not left implicit:

- **`SPEED = 541`, `INDEX_NR 1476144`.** Flagged in session 1 as implausible (indicated airspeed far
  beyond any 737 variant's structural limit) but with no equivalent first-person evidence to the `2700`
  row. Correcting it would mean guessing at a "true" value with nothing but a hunch to guess from —
  different in kind from Step 5 above, not just in confidence. Left as `541`, still flagged in the
  session 1 notebook for anyone who finds better evidence later.
- **The `OPERATOR` "unknown"-variant sweep.** Session 1 found `"UNKNOWN COMMERCIAL"` (899 rows) as a
  sibling of the literal string `"UNKNOWN"` (99,328 rows), and flagged that other equivalent variants
  might exist unchecked. Finding and consolidating all of them is a bigger, more open-ended task than
  fits inside a session already covering six other fixes — real work, not a blocker for anything planned
  for upcoming sessions.
- **The 401-group real-`FLT` residual.** Session 1's grain audit found this bucket likely mixes genuine
  multi-strike flights with the same physical strike reported more than once (the `KCLT`/American 600
  example). Separating the two needs `REMARKS` text comparison across same-day/same-flight rows — a
  project of its own, not a two-line fix, and it doesn't block the geographic/temporal EDA planned next.

## Validation

This notebook recodes, fills, and corrects — it never filters. Row count should be identical to the raw
file at every step; only the column count should move, by exactly one (the `TRANSFER` drop).

In [11]:
print('rows before:', n_rows_before, '| rows after:', len(df), '| unchanged:', n_rows_before == len(df))
print('columns before:', n_cols_before, '| columns after:', df.shape[1],
      '| change:', df.shape[1] - n_cols_before, '(expected: -1, for TRANSFER)')
assert len(df) == n_rows_before
assert df.shape[1] == n_cols_before - 1
print()
print('dtypes of every column touched this session:')
print(df[['NR_INJURIES', 'NR_FATALITIES', 'DAMAGE_LEVEL', 'NUM_SEEN', 'NUM_STRUCK',
          'SPEED', 'LATITUDE', 'LONGITUDE']].dtypes)


rows before: 354144 | rows after: 354144 | unchanged: True
columns before: 102 | columns after: 101 | change: -1 (expected: -1, for TRANSFER)

dtypes of every column touched this session:
NR_INJURIES         int64
NR_FATALITIES       int64
DAMAGE_LEVEL          str
NUM_SEEN         category
NUM_STRUCK       category
SPEED             float64
LATITUDE          float64
LONGITUDE         float64
dtype: object


## Write the cleaned dataset

**Parquet, not CSV** — deliberately. Step 4 set real ordered-categorical dtypes on `NUM_SEEN`/
`NUM_STRUCK`; CSV has no concept of that and would flatten both back to plain strings on the very next
`read_csv`, silently discarding the fix this notebook just made. Parquet stores the dtype itself, not just
the printed value, so it round-trips exactly.

**Still open, third session running.** `execution_count: null` on both cells below, same as sessions 2 and
3. Same cause each time: no `pyarrow`, no `fastparquet`, no network access to install either, in whatever
sandbox has produced this notebook. The session 3 review independently hit the identical wall trying to
run these cells itself, which rules out this being specific to one environment — it's a constraint of the
sandbox type, not a one-off. Everything feeding into these two cells *has* been executed and verified for
real, including the whitespace-normalization step and the corrected validation checks above — so there's
no reason to expect the write itself to behave differently once `pyarrow` is actually available with
network access. But "the logic checks out" and "the file exists" are different claims. Running these two
cells for real, somewhere with genuine network access — a local machine, not a sandbox like this one — is
the one step in this entire project that has to happen outside this sandbox rather than in it.

In [ ]:
import os

out_path = '../data/processed/strikes_clean.parquet'
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df.to_parquet(out_path, engine='pyarrow', index=False)
print('wrote', out_path, '--', round(os.path.getsize(out_path) / 1e6, 1), 'MB')


In [ ]:
# Round-trip check: reload from disk and confirm shape and the dtype work above survived.
df_check = pd.read_parquet('../data/processed/strikes_clean.parquet', engine='pyarrow')

print('reloaded shape:', df_check.shape, '| matches:', df_check.shape == df.shape)
print('NUM_SEEN dtype:', df_check['NUM_SEEN'].dtype, '| ordered:', df_check['NUM_SEEN'].cat.ordered)
print('NR_INJURIES dtype:', df_check['NR_INJURIES'].dtype)
print('LATITUDE dtype:', df_check['LATITUDE'].dtype)
print('TIME notna (should reflect the Step 0 fix, not the original inflated count):', df_check['TIME'].notna().sum())
print('TRANSFER present:', 'TRANSFER' in df_check.columns, '(expected: False)')


## Summary

Seven fixes applied this session — six evidenced in session 1, plus the whitespace-normalization step
added in session 3 after `TIME`'s inflated fill-rate measurement came to light: whitespace-only strings
converted to real nulls across every string column (not just `TIME`), `NR_INJURIES`/`NR_FATALITIES` filled
with 0, `TRANSFER` dropped, `DAMAGE_LEVEL`'s blank recoded to `"Unknown"`, `NUM_SEEN`/`NUM_STRUCK` cast to
ordered categoricals, one `SPEED` value corrected against direct evidence, and all 15 corrupted `LATITUDE`
values (plus the `LONGITUDE` values entangled with them) resolved or, where genuinely unrecoverable,
nulled rather than guessed at. Row count unchanged throughout; column count down by exactly one.

Three things stay open on purpose — `SPEED = 541`, the `OPERATOR` unknown-variant sweep, and the
real-`FLT` residual — each with a stated reason, not silently dropped.

**One real gap, still open across three sessions now:** the parquet write and its round-trip check are
correct, tested-up-to-that-point code that this sandbox still can't execute (no `pyarrow`, no network to
install it — confirmed again in session 4, and independently confirmed by the session 3 review running
into the identical wall in its own environment). Everything upstream checks out — including the fixes
session 3 made to how that checking is done (real null-count and range checks replacing a tautological
one, and the whitespace fix itself) — but "everything upstream checks out" is still not the same claim as
"the file exists." Running these two cells for real, in whatever local environment `pip install -r
requirements.txt` actually has network access, is the one step this project needs outside every sandbox
that has produced these notebooks so far.

**Next (session 4, continued):** run this notebook locally once `pyarrow` installs successfully, confirm
`strikes_clean.parquet` exists and the round-trip output matches what these cells already predict, then
update the README to state that as fact. **Session 5:** temporal EDA on the corrected `TIME` basis —
seasonality and time-of-day, reading from `data/processed/strikes_clean.parquet`.